# PLIP Embedding Generation with Enhanced Pre-processing

In this notebook, we will:

- Preprocess images from the **PubMed** dataset and the **matches** dataset.
- Generate embeddings for both datasets using the PLIP model.
- Save the embeddings, image paths, and captions for later use.
- Clear memory after processing to manage resources effectively.

---

## **Step 1: Import Libraries**

We begin by importing the necessary libraries.

In [1]:
# Import necessary libraries
import os
import json
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance
from tqdm import tqdm
import torch
from transformers import CLIPModel, CLIPProcessor

---

## **Step 2: Set Up Device**

We set up the device to use GPU if available, otherwise CPU.

In [2]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


---

## **Step 3: Define Helper Functions**

We define functions for image preprocessing and embedding generation.

In [3]:
# Function to preprocess images
def preprocess_image(image_path, size=(224, 224), contrast_factor=1.5):
    img = Image.open(image_path).convert("RGB")
    img = img.resize(size)
    enhancer = ImageEnhance.Contrast(img)
    img = enhancer.enhance(contrast_factor)
    return img

# Function to generate and normalize embeddings
def generate_embeddings(image_paths, processor, model, batch_size=32):
    embeddings = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc="Generating Embeddings"):
        batch_paths = image_paths[i:i+batch_size]
        images = [preprocess_image(path) for path in batch_paths]
        inputs = processor(images=images, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            outputs = model.get_image_features(**inputs)
            outputs = outputs.cpu().numpy()
            # Normalize embeddings
            outputs = outputs / np.linalg.norm(outputs, axis=1, keepdims=True)
            embeddings.append(outputs)
        # Free up memory
        del images, inputs, outputs
        torch.cuda.empty_cache()
    embeddings = np.vstack(embeddings)
    return embeddings

---

## **Step 4: Load PLIP Model and Processor**

We load the pre-trained PLIP (CLIP-based) model and the corresponding processor.

In [4]:
# Load PLIP model and processor
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

/Users/mohammedkhodorfirasal-tal/Documents/Professional/Work/Fellowship - Novartis/Data/venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


---

## **Step 5: Process PubMed Dataset**

We will:

- Load captions and UUIDs from the PubMed dataset.
- Preprocess images and collect valid image paths and captions.
- Generate embeddings for the PubMed images.

In [5]:
# -------------------------------
# Part 1: Process PubMed Dataset
# -------------------------------

# Load PubMed captions and UUIDs
pubmed_path = './pubmed_set/captions.json'
with open(pubmed_path, 'r') as file:
    data = json.load(file)
pubmed_df = pd.DataFrame(data).T
pubmed_captions = pubmed_df['caption'].tolist()
pubmed_uuids = pubmed_df['uuid'].tolist()

# Define directories
image_dir = './pubmed_set/images/'
processed_image_dir = './pubmed_set/processed_images/'
os.makedirs(processed_image_dir, exist_ok=True)

# Preprocess images and collect valid image paths
processed_image_paths = []
processed_captions = []
error_uuids = []

for uuid, caption in tqdm(zip(pubmed_uuids, pubmed_captions), desc="Processing PubMed Images", total=len(pubmed_uuids)):
    input_path = os.path.join(image_dir, uuid + '.jpg')
    output_path = os.path.join(processed_image_dir, uuid + '.jpg')
    try:
        img = preprocess_image(input_path)
        img.save(output_path)
        processed_image_paths.append(output_path)
        processed_captions.append(caption)
        img.close()
    except Exception as e:
        error_uuids.append(uuid)

print(f"Total errors encountered in PubMed dataset: {len(error_uuids)}")

Processing PubMed Images: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3309/3309 [00:14<00:00, 228.86it/s]

Total errors encountered in PubMed dataset: 37


---

## **Step 6: Generate Embeddings for PubMed Images**

We generate embeddings for the preprocessed PubMed images.

In [6]:
# Generate normalized embeddings for PubMed images
embeddings_pubmed = generate_embeddings(processed_image_paths, processor, model)

Generating Embeddings: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:56<00:00,  1.81it/s]


---

## **Step 7: Save PubMed Embeddings and Data**

We save the embeddings, image paths, and captions to disk.

In [7]:
# Save embeddings and data
np.save('embeddings_pubmed.npy', embeddings_pubmed)
np.save('paths_pubmed.npy', processed_image_paths)
np.save('captions_pubmed.npy', processed_captions)

# Clear variables to free memory
del embeddings_pubmed, processed_image_paths, processed_captions
import gc
gc.collect()

0

---

## **Step 8: Process Matches Dataset**

We will:

- Load image paths from the matches dataset.
- Generate embeddings for the matches images.

In [8]:
# -------------------------------
# Part 2: Process Matches Dataset
# -------------------------------

# Define matches directory
matches_dir = './matches/'
matches_image_files = [f for f in os.listdir(matches_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
matches_image_paths = [os.path.join(matches_dir, filename) for filename in matches_image_files]

# Generate dummy captions for matches images
matches_captions = ['' for _ in matches_image_paths]

---

## **Step 9: Generate Embeddings for Matches Images**

We generate embeddings for the matches images.

In [9]:
# Generate normalized embeddings for matches images
embeddings_matches = generate_embeddings(matches_image_paths, processor, model)

Generating Embeddings: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.51it/s]


---

## **Step 10: Save Matches Embeddings and Data**

We save the embeddings and image paths to disk.

In [10]:
# Save embeddings and data
np.save('embeddings_matches.npy', embeddings_matches)
np.save('paths_matches.npy', matches_image_paths)
np.save('captions_matches.npy', matches_captions)

# Clear variables to free memory
del embeddings_matches, matches_image_paths, matches_captions
del model, processor
torch.cuda.empty_cache()
import gc
gc.collect()

print("Embedding generation completed and data saved.")

Embedding generation completed and data saved.


---

## **Step 11: Conclusion**

We have successfully generated embeddings for both the PubMed dataset and the matches dataset and saved them for later use.

**Next Steps:**

- Proceed to the retrieval and evaluation notebook to perform image retrieval and compare results.